# Base-rate benchmark source analysis

Counts from `data/base_rate/benchmark.csv` (the benchmark definition, not merged LLM results).

In [45]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

BENCHMARK_CSV = ROOT / "data" / "base_rate" / "benchmark.csv"
if not BENCHMARK_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {BENCHMARK_CSV}. Run scripts/build_base_rate_prompts.py first."
    )

df = pd.read_csv(BENCHMARK_CSV)

print("Rows:", len(df))
print("Columns:", ", ".join([
    "vignette_name", "intersection_size", "response_type",
    "has_statistics", "variant", "scepticism_required",
]))
df[
    [
        "example_id",
        "vignette_name",
        "problem_type",
        "intersection_size",
        "response_type",
        "has_statistics",
        "variant",
        "scepticism_required",
        "scepticism_score_target",
    ]
].head()

Rows: 60
Columns: vignette_name, intersection_size, response_type, has_statistics, variant, scepticism_required


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,scepticism_required,scepticism_score_target
0,discharged_weapon_last_year__open_probs,discharged weapon (last year),well_posed,0,open,True,open_probs,False,meta
1,discharged_weapon_last_year__open_no_probs,discharged weapon (last year),well_posed,0,open,False,open_no_probs,False,meta
2,discharged_weapon_last_year__mc_numeric_probs,discharged weapon (last year),well_posed,0,mc_numeric,True,mc_numeric_probs,False,NaN
3,discharged_weapon_last_year__mc_numeric_no_probs,discharged weapon (last year),well_posed,0,mc_numeric,False,mc_numeric_no_probs,False,NaN
4,discharged_weapon_last_year__mc_full_probs,discharged weapon (last year),well_posed,0,mc_full,True,mc_full_probs,False,F|G|H


## Count by `vignette_name`

In [46]:
by_vignette = (
    df.groupby("vignette_name", observed=True)
    .size()
    .rename("n")
    .sort_index()
    .to_frame()
)
by_vignette

,n
vignette_name,
CA Trump voter,6
actor waiter overlap,6
college STEM work,6
covid vaccine (blue/red),6
diabetes insulin obese,6
discharged weapon (last year),6
english teacher humanities,6
healthcare employment,6
military overseas (federal pool),6


## Count by `response_type`

In [47]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]

by_response_type = (
    df.groupby("response_type", observed=True)
    .size()
    .rename("n")
    .reindex(RESPONSE_TYPE_ORDER)
    .to_frame()
)
by_response_type

,n
response_type,
open,20
mc_numeric,20
mc_full,20


## Count by `has_statistics`

In [48]:
by_has_statistics = (
    df.groupby("has_statistics", observed=True)
    .size()
    .rename("n")
    .to_frame()
)
by_has_statistics.index = by_has_statistics.index.map(
    {True: "probs (True)", False: "no_probs (False)"}
)
by_has_statistics

,n
has_statistics,
no_probs (False),30
probs (True),30


## Count by `intersection_size`

In [49]:
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]

by_intersection_size = (
    df.assign(intersection_size=df["intersection_size"].astype(str))
    .groupby("intersection_size", observed=True)
    .size()
    .rename("n")
    .reindex(INTERSECTION_SIZE_ORDER)
    .to_frame()
)
by_intersection_size

,n
intersection_size,
0,30
small,12
medium,6
large,12


## `intersection_size` × `response_type` × `scepticism_required`

In [50]:
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]

frame = df.assign(
    intersection_size=df["intersection_size"].astype(str),
    scepticism_required=df["scepticism_required"].astype(str).str.lower(),
)

by_intersection_response_scepticism = (
    frame.groupby(
        ["intersection_size", "response_type", "scepticism_required"],
        observed=True,
    )
    .size()
    .rename("n")
    .reset_index()
    .set_index(["intersection_size", "response_type", "scepticism_required"])
    .unstack("scepticism_required")
    .fillna(0)
    .astype(int)
)

by_intersection_response_scepticism = by_intersection_response_scepticism.reindex(
    INTERSECTION_SIZE_ORDER,
    level="intersection_size",
).reindex(RESPONSE_TYPE_ORDER, level="response_type")

by_intersection_response_scepticism

n     
scepticism_required             false true
intersection_size response_type           
0                 open             10    0
                  mc_numeric       10    0
                  mc_full          10    0
small             open              4    0
                  mc_numeric        4    0
                  mc_full           4    0
medium            open              0    2
                  mc_numeric        0    2
                  mc_full           0    2
large             open              0    4
                  mc_numeric        0    4
                  mc_full           0    4

## Cross-tab: vignette × intersection_size × response_type × has_statistics

In [51]:
cross_tab = pd.crosstab(
    [
        df["vignette_name"],
        df["intersection_size"].astype(str),
        df["response_type"],
    ],
    df["has_statistics"],
    margins=True,
)
cross_tab.columns = ["no_probs (False)", "probs (True)", "All"]
cross_tab

no_probs (False)  \
vignette_name                    intersection_size response_type                     
CA Trump voter                   0                 mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
actor waiter overlap             small             mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
college STEM work                medium            mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
covid vaccine (blue/red)         0                 mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
diabetes insulin obese           large             mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
discharged weapon (last year)    0                 mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
english teacher humanities       large             mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
healthcare employment            0                 mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
military overseas (federal pool) 0                 mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
professional drivers speeding    small             mc_full                       1   
                                                   mc_numeric                    1   
                                                   open                          1   
All                                                                             30   

                                                                  probs (True)  \
vignette_name                    intersection_size response_type                 
CA Trump voter                   0                 mc_full                   1   
                                                   mc_numeric                1   
                                                   open                      1   
actor waiter overlap             small             mc_full                   1   
                                                   mc_numeric                1   
                                                   open                      1   
college STEM work                medium            mc_full                   1   
                                                   mc_numeric                1   
                                                   open                      1   
covid vaccine (blue/red)         0                 mc_full                   1   
                                                   mc_numeric                1   
                                                   open                      1   
diabetes insulin obese           large             mc_full                   1 

## Vignette probabilities (for implausible variants)

One row per vignette from the source CSVs (`docs/base-rate-two-cause-vignettes.csv`, `docs/base-rate-overlap-vignettes.csv`).

Implausible cases will duplicate a vignette with **one** of these probabilities changed. The five columns are the rates woven into `*_probs` prompts: P(A), the C/D split under A, and P(T|C) and P(T|D) (P(T|N) is also in the prompt but omitted here).

In [52]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.build_base_rate_prompts import _load_overlap, _load_two_cause
from scripts.print_vignette_table import _short_label

PROBABILITY_COLUMNS = [
    "P_A",
    "P_C_given_A",
    "P_D_given_A",
    "P_T_given_C",
    "P_T_given_D",
]
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]


def diversity_first_vignette_names(names: pd.Series, sizes: pd.Series) -> list[str]:
    """Round-robin by intersection_size so the first N rows cover every size."""
    size_text = sizes.astype(str)
    by_size = {
        size: sorted(names[size_text == size].unique())
        for size in INTERSECTION_SIZE_ORDER
    }
    ordered: list[str] = []
    for round_idx in range(max(len(bucket) for bucket in by_size.values())):
        for size in INTERSECTION_SIZE_ORDER:
            bucket = by_size[size]
            if round_idx < len(bucket):
                ordered.append(bucket[round_idx])
    return ordered


def vignette_name_rank(names: pd.Series, sizes: pd.Series) -> dict[str, int]:
    return {
        name: rank
        for rank, name in enumerate(diversity_first_vignette_names(names, sizes))
    }


def sort_vignette_rows(frame: pd.DataFrame) -> pd.DataFrame:
    rank = vignette_name_rank(frame["vignette_name"], frame["intersection_size"])
    ordered = frame.assign(_vignette_order=frame["vignette_name"].map(rank))
    if "parameter" in ordered.columns:
        ordered = ordered.assign(
            _parameter_order=pd.Categorical(
                ordered["parameter"], categories=PROBABILITY_COLUMNS, ordered=True
            )
        )
        sort_cols = ["_vignette_order", "_parameter_order"]
    else:
        sort_cols = ["_vignette_order", "vignette_name"]
    return (
        ordered.sort_values(sort_cols)
        .drop(columns=[c for c in ordered.columns if c.startswith("_")])
        .reset_index(drop=True)
    )


def vignette_probability_row(vignette) -> dict:
    return {
        "vignette_name": vignette.name,
        "problem_type": "well_posed" if vignette.well_posed else "overlap",
        "intersection_size": vignette.intersection_size,
        "C": _short_label(vignette.c, max_len=35),
        "D": _short_label(vignette.d, max_len=35),
        "P_A": vignette.p_a,
        "P_C_given_A": vignette.q_c,
        "P_D_given_A": vignette.q_d,
        "P_T_given_C": vignette.s_c,
        "P_T_given_D": vignette.s_d,
    }


vignettes = _load_two_cause() + _load_overlap()
vignette_probs = pd.DataFrame(vignette_probability_row(v) for v in vignettes)
vignette_probs = sort_vignette_rows(vignette_probs)

vignette_probs_long = sort_vignette_rows(
    vignette_probs.melt(
        id_vars=[
            "vignette_name",
            "problem_type",
            "intersection_size",
            "C",
            "D",
        ],
        value_vars=PROBABILITY_COLUMNS,
        var_name="parameter",
        value_name="value",
    )
)

len(INTERSECTION_SIZE_ORDER)  # first N vignettes cover every intersection_size
vignette_probs

,vignette_name,problem_type,intersection_size,C,D,P_A,P_C_given_A,P_D_given_A,P_T_given_C,P_T_given_D
0,diabetes insulin obese,overlap,large,Uses insulin among adults with dia…,Has obesity among adults with diag…,0.11300,0.282,0.471,0.200,0.160
1,english teacher humanities,overlap,large,Main teaching assignment is Englis…,Bachelor's degree major field is E…,0.00760,0.148,0.171,0.690,0.550
2,college STEM work,overlap,medium,STEM field of study among first-ge…,Employed while enrolled among firs…,0.25800,0.180,0.660,0.850,0.740
3,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,0.00034,0.035,0.298,0.650,0.550
4,professional drivers speeding,overlap,small,Heavy and tractor-trailer truck dr…,Bus driver among professional driv…,0.02400,0.534,0.145,0.160,0.100
5,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,0.13000,0.380,0.600,0.310,0.270
6,covid vaccine (blue/red),well_posed,0,Blue-state resident among A,Red-state resident among A,0.27000,0.710,0.290,0.080,0.100
7,discharged weapon (last year),well_posed,0,Urban police officer,Rural/small-jurisdiction police of…,0.44000,0.680,0.300,0.003,0.002
8,healthcare employment,well_posed,0,Physician,Non-physician health care professi…,0.11000,0.100,0.900,0.540,0.600
9,military overseas (federal pool),well_posed,0,US Army active-duty service member,US Navy or US Air Force active-dut…,0.40000,0.350,0.510,0.580,0.640


In [53]:
vignette_probs_long

,vignette_name,problem_type,intersection_size,C,D,parameter,value
5,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_A,0.13000
15,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_C_given_A,0.38000
25,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_D_given_A,0.60000
35,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_T_given_C,0.31000
45,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_T_given_D,0.27000
3,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_A,0.00034
13,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_C_given_A,0.03500
23,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_D_given_A,0.29800
33,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_T_given_C,0.65000
43,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_T_given_D,0.55000


## Implausible statistics

One row per `(vignette, parameter)` with a baseline `value` and an `implausible_value` to fill in by hand (default `N/A`).

Stored in `data/base_rate/implausible_statistics.csv` so edits survive notebook restarts.

In [54]:
IMPLAUSIBLE_CSV = ROOT / "data" / "base_rate" / "implausible_statistics.csv"

implausible_statistics = vignette_probs_long.copy()
merge_keys = ["vignette_name", "parameter"]

if IMPLAUSIBLE_CSV.is_file():
    saved = pd.read_csv(IMPLAUSIBLE_CSV)
    implausible_statistics = implausible_statistics.merge(
        saved[merge_keys + ["implausible_value"]],
        on=merge_keys,
        how="left",
    )
    implausible_statistics["implausible_value"] = implausible_statistics[
        "implausible_value"
    ].fillna("N/A")
else:
    implausible_statistics["implausible_value"] = "N/A"
    IMPLAUSIBLE_CSV.parent.mkdir(parents=True, exist_ok=True)

implausible_statistics = sort_vignette_rows(implausible_statistics)
if not IMPLAUSIBLE_CSV.is_file():
    implausible_statistics.to_csv(IMPLAUSIBLE_CSV, index=False)

implausible_statistics

,vignette_name,problem_type,intersection_size,C,D,parameter,value,implausible_value
0,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_A,0.13000,0.8
1,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_C_given_A,0.38000,N/A
2,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_D_given_A,0.60000,N/A
3,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_T_given_C,0.31000,N/A
4,CA Trump voter,well_posed,0,Other California registrant,Southern California registrant,P_T_given_D,0.27000,N/A
5,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_A,0.00034,N/A
6,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_C_given_A,0.03500,0.7
7,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_D_given_A,0.29800,N/A
8,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_T_given_C,0.65000,N/A
9,actor waiter overlap,overlap,small,Also holds a waiter/waitress occup…,Actor occupation is the secondary …,P_T_given_D,0.55000,N/A


In [55]:
def _implausible_is_set(series: pd.Series) -> pd.Series:
    as_text = series.astype("string")
    return as_text.notna() & ~as_text.str.upper().eq("N/A") & as_text.str.strip().ne("")


filled = implausible_statistics[_implausible_is_set(implausible_statistics["implausible_value"])]
counts = filled.groupby("vignette_name", sort=True).size()
all_vignettes = implausible_statistics.drop_duplicates("vignette_name")[
    "vignette_name"
].tolist()

validation = pd.DataFrame({"vignette_name": all_vignettes})
validation["implausible_count"] = validation["vignette_name"].map(counts).fillna(0).astype(int)
validation["status"] = validation["implausible_count"].map(
    lambda n: "ok" if n == 1 else ("missing" if n == 0 else "too many")
)

chosen = filled[["vignette_name", "parameter", "value", "implausible_value"]].rename(
    columns={"value": "baseline_value"}
)
validation = validation.merge(chosen, on="vignette_name", how="left")

problems = validation[validation["status"] != "ok"]
if not problems.empty:
    display(problems)
    raise ValueError(
        "Each vignette must have exactly one implausible_value set; "
        f"found {len(problems)} problem(s)."
    )

validation

,vignette_name,implausible_count,status,parameter,baseline_value,implausible_value
0,CA Trump voter,1,ok,P_A,0.130,0.8
1,actor waiter overlap,1,ok,P_C_given_A,0.035,0.7
2,college STEM work,1,ok,P_D_given_A,0.660,0.2
3,covid vaccine (blue/red),1,ok,P_T_given_C,0.080,0.8
4,diabetes insulin obese,1,ok,P_T_given_D,0.160,0.8
5,discharged weapon (last year),1,ok,P_A,0.440,0.1
6,english teacher humanities,1,ok,P_T_given_C,0.690,0.1
7,healthcare employment,1,ok,P_T_given_C,0.540,0.05
8,military overseas (federal pool),1,ok,P_T_given_D,0.640,0.98
9,professional drivers speeding,1,ok,P_T_given_C,0.160,0.9


In [56]:
import re
from dataclasses import replace

from scripts.build_base_rate_prompts import narrative_with_probs

PROMPT_PCT_RE = re.compile(r"(\d+(?:\.\d+)?)%")
IMPLAUSIBLE_PARAM_FIELD = {
    "P_A": "p_a",
    "P_C_given_A": "q_c",
    "P_D_given_A": "q_d",
    "P_T_given_C": "s_c",
    "P_T_given_D": "s_d",
}


def prompt_percentages(prompt: str) -> list[float]:
    return [float(match) / 100 for match in PROMPT_PCT_RE.findall(prompt)]


def vignette_with_implausible(vignette, parameter: str, value: float):
    return replace(vignette, **{IMPLAUSIBLE_PARAM_FIELD[parameter]: float(value)})


vignette_by_name = {v.name: v for v in vignettes}
filled = implausible_statistics[
    _implausible_is_set(implausible_statistics["implausible_value"])
]

prompt_checks = []
for _, row in filled.iterrows():
    base = vignette_by_name[row["vignette_name"]]
    modified = vignette_with_implausible(
        base, row["parameter"], row["implausible_value"]
    )
    orig_pcts = prompt_percentages(narrative_with_probs(base))
    impl_pcts = prompt_percentages(narrative_with_probs(modified))

    if len(orig_pcts) != len(impl_pcts):
        changed_count = None
        prompt_change = f"length {len(orig_pcts)} -> {len(impl_pcts)}"
        status = "length mismatch"
    else:
        changed_idx = [
            i
            for i, (before, after) in enumerate(zip(orig_pcts, impl_pcts))
            if abs(before - after) > 1e-9
        ]
        changed_count = len(changed_idx)
        if changed_count == 1:
            i = changed_idx[0]
            prompt_change = f"{orig_pcts[i] * 100:g}% -> {impl_pcts[i] * 100:g}%"
            status = "ok"
        else:
            prompt_change = str(changed_idx)
            status = "not one change"

    prompt_checks.append(
        {
            "vignette_name": row["vignette_name"],
            "parameter": row["parameter"],
            "changed_probability_count": changed_count,
            "prompt_change": prompt_change,
            "status": status,
        }
    )

prompt_validation = pd.DataFrame(prompt_checks)
problems = prompt_validation[prompt_validation["status"] != "ok"]
if not problems.empty:
    display(problems)
    raise ValueError(
        "Each implausible vignette should change exactly one number in the "
        "*_probs prompt prose; "
        f"found {len(problems)} problem(s)."
    )

prompt_validation

,vignette_name,parameter,changed_probability_count,prompt_change,status
0,CA Trump voter,P_A,1,13% -> 80%,ok
1,actor waiter overlap,P_C_given_A,1,3.5% -> 70%,ok
2,college STEM work,P_D_given_A,1,66% -> 20%,ok
3,covid vaccine (blue/red),P_T_given_C,1,8% -> 80%,ok
4,diabetes insulin obese,P_T_given_D,1,16% -> 80%,ok
5,discharged weapon (last year),P_A,1,44% -> 10%,ok
6,english teacher humanities,P_T_given_C,1,69% -> 10%,ok
7,healthcare employment,P_T_given_C,1,54% -> 5%,ok
8,military overseas (federal pool),P_T_given_D,1,64% -> 98%,ok
9,professional drivers speeding,P_T_given_C,1,16% -> 90%,ok
